# Site analytics

Every page view on lisadlima.com lands in `/app/data/analytics.jsonl` (one JSON object per line, written by the middleware in `app.ipynb`). This notebook answers the recurring questions; add cells as new questions come up. The quick-glance version of this lives at `/stats?key=...` on the site itself.

In [ ]:
import pandas as pd, json
rows = [json.loads(l) for l in open('/app/data/analytics.jsonl')]
df = pd.DataFrame(rows)
df['ts'] = pd.to_datetime(df['ts'])
df['day'] = df['ts'].dt.date
BOTS = ('bot','crawl','spider','gpt','claude','perplexity','slurp','facebookexternalhit','slack','discord','whatsapp','curl','python-httpx','wget')
df['is_bot'] = df['ua'].str.lower().str.contains('|'.join(BOTS), regex=True)
human = df[~df['is_bot']]
print(f'{len(df)} hits total: {len(human)} human, {df.is_bot.sum()} bot')
df.tail(3)

## Views and visitors by day

In [ ]:
human.groupby('day').agg(views=('path','size'), visitors=('v','nunique'))

## What do people look at?

In [ ]:
human['path'].value_counts().head(15)

## Where do they come from?

In [ ]:
human[human['ref']!='']['ref'].value_counts().head(15)

## Does anyone reach the contact page? (the funnel question)

In [ ]:
contact_visitors = set(human[human['path']=='/contact']['v'])
print(f'{len(contact_visitors)} visitors reached /contact')
human[human['v'].isin(contact_visitors)].groupby('v')['path'].apply(list).head(10)

## Who finds the robot?

In [ ]:
print(f"robot page views: {len(human[human['path']=='/robot'])}")

## Do AI crawlers fetch llms.txt?

In [ ]:
df[df['path']=='/llms.txt'][['ts','ua']].tail(15)